# SfM robustness: GPU-only experiments (Colab)

Runs the parts that need a GPU and Hugging Face models; everything else runs on CPU elsewhere.

1. **E2b diffusion generative noise**: SD-Turbo img2img at strength 0.1 to 0.6 on every view.
2. **E3 diffusion super-resolution**: SD x4 upscaler, compared against Real-ESRGAN and bicubic.
3. **E6 learned features**: ALIKED+LightGlue vs SIFT, with incremental (COLMAP) vs global (GLOMAP) mapping.

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. Results are appended to
`MyDrive/sfmrob_results/` after every run, so if Colab disconnects, just run all again and it resumes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/sfmrob_results'
import os; os.makedirs(OUT, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!rm -rf code && git clone -q https://github.com/amoghshetty7436/Computer-Vision code
%cd /content/code
!pip -q install pycolmap kornia diffusers transformers accelerate opencv-python-headless
import os
os.environ['SFMROB_DATA'] = '/content/data'
os.environ['SFMROB_WORK'] = '/content/work'
os.environ['GLOG_minloglevel'] = '2'

In [ ]:
# ALIKED weights (kornia's default URL is sometimes blocked; fetch from raw.githubusercontent)
import torch, urllib.request, pathlib
ck = pathlib.Path(torch.hub.get_dir()) / 'checkpoints'; ck.mkdir(parents=True, exist_ok=True)
if not (ck / 'aliked-n16.pth').exists():
    urllib.request.urlretrieve('https://raw.githubusercontent.com/Shiaoming/ALIKED/main/models/aliked-n16.pth', ck / 'aliked-n16.pth')

In [ ]:
# Same scene subsets as the CPU runs: gerrard-hall 50 images, south-building 40 images, 1024 px.
import subprocess, sys
SCENES = [('gerrard-hall', 50), ('south-building', 40)]
def run(exp, scene, n, only, seeds, full=True):
    cmd = [sys.executable, '-m', 'sfmrob.run', '--exp', exp, '--scene', scene, '--n', str(n), '--size', '1024',
           '--only', only, '--seeds', *map(str, seeds), '--out', OUT, '--threads', '2'] + (['--full'] if full else [])
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd='/content/code',
                         env={**os.environ, 'PYTHONPATH': '/content/code'})
    for line in p.stdout:
        if line[:1] not in 'IWE' or not line[1:9].isdigit():
            print(line, end='')
    p.wait()

In [ ]:
# 0) sanity check: save a strip of diffusion outputs to Drive (clean | strength 0.1 | 0.3 | 0.6)
import numpy as np, sys
sys.path.insert(0, '/content/code')
from pathlib import Path
from PIL import Image
from sfmrob import data as D
from sfmrob.perturb.diffusion import sdturbo_img2img
names = D.prepare_colmap_subset('gerrard-hall', 50, 1024, Path('/content/work/gerrard-hall_50_1024/clean'))
img = np.asarray(Image.open(f'/content/work/gerrard-hall_50_1024/clean/{names[10]}').convert('RGB'))
outs = [img] + [sdturbo_img2img(img, s, np.random.default_rng(0)) for s in (1, 3, 5)]
for o in outs: print(o.shape, o.mean().round(1))
Image.fromarray(np.hstack(outs)).resize((2048, 342)).save(f'{OUT}/sdturbo_sanity.jpg')

In [ ]:
# 1a) GAN / LaMa conditions (Real-ESRGAN regeneration, LaMa inpainting, ESRGAN super-res, LaMa crowd removal).
# These take ~20 min per run on CPU, a minute or two on a T4, so they run here. Most important cell: run it first.
for seed in [0, 1, 2]:
    for scene, n in SCENES:
        run('e2_generative', scene, n, 'regen_esrgan', [seed], full=False)
        run('e2_generative', scene, n, 'inpaint_lama', [seed], full=False)
        run('e3_superres', scene, n, 'sr_realesrgan', [seed], full=False)
        run('e4_crowds', scene, n, 'lama_remove', [seed], full=False)
        run('e5_contamination', scene, n, 'inpaint_lama', [seed], full=False)


In [ ]:
# 1) diffusion generative noise (~1-2 h)
for scene, n in SCENES:
    run('e2_generative', scene, n, 'sdturbo_img2img', [0, 1, 2])

In [ ]:
# 2) diffusion super-resolution (~20 min)
for scene, n in SCENES:
    run('e3_superres', scene, n, 'sr_sd_x4', [0])

In [ ]:
# 3) ALIKED+LightGlue, incremental and global mapping (~1.5 h)
for scene, n in SCENES:
    run('e6_methods', scene, n, 'aliked_lg-', [0], full=False)

In [ ]:
!ls -la {OUT}